# Implementing GEORGE to Remedy Spurious Correlations in SpuCoMNIST

In [ ]:
!pip install spuco


In [7]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## Step 1: ERM Baseline Training

Train a standard ERM model on SpuCoMNIST with 99% spurious correlation. This model learns the spurious feature (color) rather than the true feature (digit shape), so it will fail on minority groups where the spurious and class labels don't match.

In [ ]:
from spuco.datasets import SpuCoMNIST, SpuriousFeatureDifficulty
import torchvision.transforms as T

difficulty = SpuriousFeatureDifficulty.MAGNITUDE_LARGE

classes = [[0, 1], [2, 3], [4, 5], [6, 7], [8, 9]]

trainset = SpuCoMNIST(
    root="/data/mnist/",
    spurious_feature_difficulty=difficulty,
    spurious_correlation_strength=0.99,
    classes=classes,
    split="train"
)
trainset.initialize()

valset = SpuCoMNIST(
    root="/data/mnist/",
    spurious_feature_difficulty=difficulty,
    classes=classes,
    split="val"
)
valset.initialize()


testset = SpuCoMNIST(
    root="/data/mnist/",
    spurious_feature_difficulty=difficulty,
    classes=classes,
    split="test"
)
testset.initialize()


In [9]:
print(len(valset))
print(len(trainset))
print(len(testset))

11996
48004
10000


## Creating Data Loaders

In [10]:
from torch.utils.data import DataLoader

train_dl = DataLoader(trainset, batch_size=64, shuffle=True, num_workers=2)
valid_dl = DataLoader(valset, batch_size=64, num_workers=2)

## Initializing Model and Training Loop

In [14]:
from spuco.models import model_factory
from spuco.robust_train import ERM
from torch.optim import SGD
import torch.nn as nn



#model_factory(arch: str, input_shape: Tuple[int, int, int], num_classes: int, pretrained: bool = True)
model = model_factory("lenet", trainset[0][0].shape, trainset.num_classes).to(device) #using pretrained lenet
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
num_epochs=3
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

for epoch in range(num_epochs):
    model.train() #training mode
    running_loss=0
    for xb, yb in train_dl: #goes through the images and masks
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad() #zero the gradients each batch
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(train_dl)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.3f}")



Epoch 1/3, Loss: 0.244
Epoch 2/3, Loss: 0.068
Epoch 3/3, Loss: 0.060


## ERM Evaluation

With 99% spurious correlation, ERM achieves near-perfect accuracy on groups where the spurious feature matches the class label (diagonal groups), but collapses to ~0% on minority groups where they don't align.

In [ ]:
from spuco.evaluate import Evaluator

#using spuco function
evaluator = Evaluator(
    testset=testset,
    group_partition=testset.group_partition,
    group_weights=trainset.group_weights,
    batch_size=64,
    model=model,
    device=device,
    verbose=True
)
evaluator.evaluate()

## Saving ERM Model Weights

In [ ]:
torch.save(model.state_dict(), "erm2_weights.pth")

## Step 2: Infer Spurious Groups via Clustering

GEORGE uses K-Means to cluster the ERM model's output logits within each class. The intuition: the model is highly confident on majority-group examples (learned via spurious feature) and uncertain on minority-group examples. Clustering by confidence separates them without needing group labels.

In [18]:
from spuco.group_inference import Cluster, ClusterAlg
#first collecting the outputs
model.eval()
all_outputs = []
with torch.no_grad():
    for xb, _ in train_dl:
        xb = xb.to(device)
        preds = model(xb)
        all_outputs.append(preds.cpu())
logits = torch.cat(all_outputs, dim=0)
cluster = Cluster(
    Z=logits,
    class_labels=trainset.labels,
    cluster_alg=ClusterAlg.KMEANS, #k means method for clustering
    num_clusters=2,
    device=device,
    verbose=True
)
group_partition = cluster.infer_groups()

Clustering class-wise: 100%|██████████| 5/5 [00:00<00:00, 17.86it/s]


In [19]:
for key in sorted(group_partition.keys()):
    print(key, len(group_partition[key]))

(0, 0) 4024
(0, 1) 6109
(1, 0) 3831
(1, 1) 5841
(2, 0) 3568
(2, 1) 5443
(3, 0) 5829
(3, 1) 3918
(4, 0) 5641
(4, 1) 3800


## Step 3: Retrain with Group-Balanced Batches

With inferred group labels from Step 2, retrain using `GroupBalanceBatchERM`, which samples equally from each inferred group per batch. This counteracts the majority group's overrepresentation that caused ERM to latch onto the spurious feature.

In [ ]:
from spuco.robust_train import GroupBalanceBatchERM
model = model_factory("lenet", trainset[0][0].shape, trainset.num_classes).to(device) #using pretrained lenet

group_balance = GroupBalanceBatchERM(
    model=model,
    num_epochs=3,
    trainset=trainset,
    group_partition=group_partition,
    batch_size=64,
    optimizer=SGD(model.parameters(), lr=0.01,  momentum=0.9),
    device=device,
    verbose=True
)
group_balance.train()